# 2.06 随机近似：增量更新的数学基础

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/02_rl/2.06_stochastic_approximation.ipynb)


## 为什么要学

MC 方法要等收集完所有数据才求平均。能不能**来一个样本就更新一次**？可以，而且这背后有一套严格的理论：**随机近似（Stochastic Approximation）**。

TD 学习、Q-learning、SGD 都是随机近似算法。理解了它，你就明白为什么 RL 的更新公式都长成"**新估计 = 旧估计 − 步长 × (旧估计 − 目标)**"这个样子。对应书中第 6 章。

---

## 1. 增量式求均值

$N$ 个样本的均值 $w_{k}=\frac{1}{k}\sum_{i=1}^{k}x_i$ 可以增量计算（1.01 课的 `RunningMean`）：

$$
w_{k+1} = w_k - \frac{1}{k}\left(w_k - x_k\right)
$$

把 $\frac1k$ 换成一般的步长 $\alpha_k$：

$$
w_{k+1} = w_k - \alpha_k\left(w_k - x_k\right)
$$

只要 $\alpha_k$ 满足一定条件，$w_k$ 依然收敛到 $\mathbb{E}[X]$。**这就是 TD 更新的雏形**：$x_k$ 换成 TD 目标 $r+\gamma v(s')$。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
xs = rng.normal(5.0, 3.0, size=2000)     # 真实均值 5
plt.figure(figsize=(6, 3.5))
for name, alpha_fn in [("alpha=1/k", lambda k: 1 / k), ("alpha=0.1 (const)", lambda k: 0.1),
                       ("alpha=0.01 (const)", lambda k: 0.01), ("alpha=1/sqrt(k)", lambda k: 1 / np.sqrt(k))]:
    w, hist = 0.0, []
    for k, x in enumerate(xs, start=1):
        w = w - alpha_fn(k) * (w - x)
        hist.append(w)
    plt.plot(hist, label=name)
plt.axhline(5, color="k", ls="--"); plt.ylim(0, 8); plt.legend(); plt.xlabel("k"); plt.title("incremental mean estimation"); plt.show()

观察：

- $\alpha=1/k$：精确的样本均值，平稳收敛；
- 常数步长 0.1：很快到达附近，但**一直在抖**（因为最近的样本权重始终很大）；
- 常数步长 0.01：抖得少，但收敛慢。

**常数步长**不满足收敛条件，但它能**跟踪变化的目标**（非平稳问题）。深度 RL 中几乎都用常数学习率，因为目标（策略）一直在变。

---

## 2. Robbins-Monro（RM）算法

**问题**：求方程 $g(w)=0$ 的根，但我们**不知道** $g$ 的表达式，只能得到带噪声的观测 $\tilde g(w,\eta)=g(w)+\eta$。

**RM 算法**：

$$
w_{k+1} = w_k - a_k\,\tilde g(w_k,\eta_k)
$$

**RM 定理**（简化版）：若 $g$ 单调递增且导数有界、噪声均值为 0、方差有界，且步长满足

$$
\sum_{k=1}^\infty a_k=\infty,\qquad\sum_{k=1}^\infty a_k^2<\infty
$$

则 $w_k$ 以概率 1 收敛到根 $w^*$。

两个条件的直觉：

- $\sum a_k^2<\infty$：步长要**最终趋于 0**，否则噪声会让 $w$ 永远抖动；
- $\sum a_k=\infty$：步长又**不能衰减太快**，否则还没走到根就停下了。

$a_k=1/k$ 同时满足两者；$a_k=1/k^2$ 不满足第一个；常数步长不满足第二个。

**增量求均值是 RM 的特例**：要求 $\mathbb{E}[X]$，等价于求 $g(w)=w-\mathbb{E}[X]=0$ 的根，观测 $\tilde g = w - x_k$。

### 练习 1：用 RM 算法求根

$g(w)=\tanh(w-1)$（根为 $w^*=1$），每次观测带噪声 $\eta\sim\mathcal{N}(0, 0.5^2)$。

In [ ]:
def robbins_monro(g_noisy, w0, n_steps, a_fn):
    """返回 w 的历史（长度 n_steps+1）"""
    # TODO: w_{k+1} = w_k - a_k * g_noisy(w_k)，k 从 1 开始
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def robbins_monro(g_noisy, w0, n_steps, a_fn):
    """返回 w 的历史（长度 n_steps+1）"""
    w, hist = w0, [w0]
    for k in range(1, n_steps + 1):
        w = w - a_fn(k) * g_noisy(w)
        hist.append(w)
    return np.array(hist)

In [ ]:
# ✅ 自测
rng = np.random.default_rng(1)
g_noisy = lambda w: np.tanh(w - 1) + rng.normal(0, 0.5)
hist = robbins_monro(g_noisy, w0=3.0, n_steps=3000, a_fn=lambda k: 1 / k)
hist2 = robbins_monro(g_noisy, w0=3.0, n_steps=3000, a_fn=lambda k: 1 / k ** 2)
print(f"a_k=1/k  : 最终 w={hist[-1]:.3f}")
print(f"a_k=1/k^2: 最终 w={hist2[-1]:.3f}  （步长衰减太快，卡在半路）")
assert abs(hist[-1] - 1) < 0.15 and abs(hist2[-1] - 1) > 0.3
plt.figure(figsize=(5, 3)); plt.plot(hist, label="a=1/k"); plt.plot(hist2, label="a=1/k^2"); plt.axhline(1, c="k", ls="--")
plt.legend(); plt.title("Robbins-Monro"); plt.show()
print("通过 ✅")

---

## 3. 随机梯度下降（SGD）是 RM 算法

优化问题 $\min_w J(w)=\mathbb{E}[f(w,X)]$ 的最优点满足 $\nabla J(w)=\mathbb{E}[\nabla_w f(w,X)]=0$，这是一个求根问题！用单个样本的梯度 $\nabla_w f(w,x_k)$ 作为带噪观测，RM 算法就变成了 SGD：

$$
w_{k+1}=w_k-\alpha_k\nabla_w f(w_k,x_k)
$$

| 方法 | 每步用多少样本 | 特点 |
|---|---|---|
| 批量梯度下降（BGD） | 全部 $N$ 个 | 准确但慢 |
| 随机梯度下降（SGD） | 1 个 | 快但噪声大 |
| 小批量（MBGD） | $m$ 个 | 深度学习的标准做法 |

### 练习 2：用 SGD 估计均值

均值估计可以写成优化问题 $\min_w\mathbb{E}\left[\frac12(w-X)^2\right]$。写出 SGD（每步 1 个样本）和小批量（每步 $m$ 个）版本，与增量均值比较。

In [ ]:
def sgd_mean(xs, alpha_fn, batch=1):
    """返回 w 的历史"""
    # TODO: 梯度 ∇_w ½(w-x)² = w - x；小批量就对 batch 个样本的梯度取平均
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def sgd_mean(xs, alpha_fn, batch=1):
    """返回 w 的历史"""
    w, hist = 0.0, []
    for k, start in enumerate(range(0, len(xs), batch), start=1):
        xb = xs[start : start + batch]
        grad = np.mean(w - xb)
        w = w - alpha_fn(k) * grad
        hist.append(w)
    return np.array(hist)

In [ ]:
# ✅ 自测
h1 = sgd_mean(xs, lambda k: 1 / k, batch=1)
assert np.isclose(h1[-1], xs.mean())        # α=1/k 的 SGD 恰好就是增量均值！
h_const = sgd_mean(xs, lambda k: 0.05, batch=1)
h_mb = sgd_mean(xs, lambda k: 0.05, batch=20)
print(f"真实均值 5；SGD(1/k) {h1[-1]:.3f}；SGD(0.05) {h_const[-1]:.3f}；小批量20(0.05) {h_mb[-1]:.3f}")
print("最后 50 步的标准差：单样本", np.std(h_const[-50:]).round(3), " 小批量", np.std(h_mb[-50:]).round(3))
print("通过 ✅ —— 小批量平均降低了梯度噪声")

---

## 4. 本课与 RL 的联系

所有 RL 增量更新都是这个模板：

$$
\underbrace{\text{新估计}}_{w_{k+1}} = \underbrace{\text{旧估计}}_{w_k} - \underbrace{\alpha_k}_{\text{步长}}\big(\underbrace{\text{旧估计} - \text{目标}}_{\text{误差}}\big)
$$

| 算法 | 估计什么 | "目标"是什么 |
|---|---|---|
| 增量均值 | $\mathbb{E}[X]$ | 样本 $x_k$ |
| MC | $q_\pi(s,a)$ | 回报 $g_t$ |
| TD(0) | $v_\pi(s)$ | $r_{t+1}+\gamma v(s_{t+1})$ |
| Sarsa | $q_\pi(s,a)$ | $r_{t+1}+\gamma q(s_{t+1},a_{t+1})$ |
| Q-learning | $q^*(s,a)$ | $r_{t+1}+\gamma\max_{a}q(s_{t+1},a)$ |

## 小结

- 增量更新 $w\leftarrow w-\alpha(w-x)$ 来一个样本更新一次。
- RM 定理：步长满足 $\sum a_k=\infty,\ \sum a_k^2<\infty$ 时收敛。
- SGD 是 RM 的特例；小批量降低梯度方差。
- 下一课：把"目标"换成 $r+\gamma v(s')$，就得到 TD 学习。